# 03 描述统计、可视化与分析

In [ ]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['font.family'] = 'DejaVu Sans'
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.dpi'] = 150

panel = pd.read_parquet('data/combined/csmar_firm_year_panel.parquet')
print(f"面板数据: {len(panel)} 行, {panel['code'].nunique()} 家公司, {panel['year'].min()}-{panel['year'].max()} 年")


## 4.1 年度描述统计

In [ ]:
yearly = pd.read_csv('output/tables/yearly_summary.csv', encoding='utf-8-sig')
lev_tbl = yearly[yearly['variable']=='Lev'][['year','mean','median','std','min','max','n']]
print("Lev 年度统计（前10行）:")
print(lev_tbl.head(10).to_string(index=False))


## 4.2 图1：Lev 均值和中位数时序图

In [ ]:
lev_yr = yearly[yearly['variable']=='Lev'].sort_values('year')

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(lev_yr['year'], lev_yr['mean'], 'b-o', markersize=4, linewidth=1.8, label='Mean Lev')
ax.plot(lev_yr['year'], lev_yr['median'], 'r--s', markersize=4, linewidth=1.8, label='Median Lev')
ax.set_xlabel('Year', fontsize=12)
ax.set_ylabel('Leverage Ratio (Lev)', fontsize=12)
ax.set_title('Figure 1: Annual Mean and Median of Total Leverage (Lev)', fontsize=13)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
ax.xaxis.set_major_locator(mticker.MultipleLocator(2))
plt.tight_layout()
plt.savefig('output/figures/fig01_lev_mean_median.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: output/figures/fig01_lev_mean_median.png")


**图1解读**：均值持续高于中位数，表明负债率分布呈右偏，少数高杠杆公司拉高了均值。2003-2013年均值较高（约0.50-0.55），与信贷扩张周期吻合；2015年后随去杠杆推进有所回落。均值长期高于中位数时，中位数更能代表典型公司的财务特征。

## 4.2 图2：ROA 和 Cash 均值时序图

In [ ]:
roa_yr = yearly[yearly['variable']=='ROA'].sort_values('year')
cash_yr = yearly[yearly['variable']=='Cash'].sort_values('year')

fig, ax1 = plt.subplots(figsize=(10, 5))
c1, c2 = '#1f77b4', '#ff7f0e'
ax1.plot(roa_yr['year'], roa_yr['mean'], '-o', color=c1, markersize=4, linewidth=1.8, label='ROA (left)')
ax1.set_xlabel('Year', fontsize=12)
ax1.set_ylabel('ROA', fontsize=12, color=c1)
ax1.tick_params(axis='y', labelcolor=c1)

ax2 = ax1.twinx()
ax2.plot(cash_yr['year'], cash_yr['mean'], '--s', color=c2, markersize=4, linewidth=1.8, label='Cash (right)')
ax2.set_ylabel('Cash Ratio', fontsize=12, color=c2)
ax2.tick_params(axis='y', labelcolor=c2)

lines = ax1.get_lines() + ax2.get_lines()
ax1.legend(lines, [l.get_label() for l in lines], fontsize=11)
ax1.set_title('Figure 2: Annual Mean of ROA and Cash Ratio', fontsize=13)
ax1.grid(True, alpha=0.3)
ax1.xaxis.set_major_locator(mticker.MultipleLocator(2))
plt.tight_layout()
plt.savefig('output/figures/fig02_roa_cash_mean.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: output/figures/fig02_roa_cash_mean.png")


**图2解读**：ROA在2007-2008年达到峰值，随后受金融危机冲击大幅下滑，2012-2015年持续低迷。Cash ratio呈缓慢上升趋势，与预防性储蓄动机增强有关。仅凭均值趋势不能推断因果关系，需要控制行业、规模等因素后进行面板回归分析。

## 第五部分：行业负债率特征分析

In [ ]:
target_inds = {'C':'Manufacturing','D':'Power/Gas/Water','G':'Transport/Storage',
               'E':'Construction','K':'Real Estate','F':'Wholesale/Retail','J':'Finance'}

ind_panel = panel[panel['industry_code'].isin(target_inds.keys())].copy()
ind_panel = ind_panel.dropna(subset=['Lev','total_assets'])
eq_wt = ind_panel.groupby(['industry_code','year'])['Lev'].mean().reset_index()
eq_wt.columns = ['industry_code','year','Lev_eq']

colors = plt.cm.tab10(range(len(target_inds)))
styles = ['-o','-s','-^','-D','-v','-P','-*']

# 图3
fig, ax = plt.subplots(figsize=(12, 6))
for i, (code, label) in enumerate(target_inds.items()):
    sub = eq_wt[eq_wt['industry_code']==code].sort_values('year')
    ax.plot(sub['year'], sub['Lev_eq'], styles[i], color=colors[i], markersize=3, linewidth=1.5, label=f'{code}: {label}')
ax.set_xlabel('Year', fontsize=12)
ax.set_ylabel('Equal-Weighted Mean Lev', fontsize=12)
ax.set_title('Figure 3: Industry Leverage Ratio (Equal-Weighted)', fontsize=13)
ax.legend(fontsize=9, loc='upper left', bbox_to_anchor=(1,1))
ax.grid(True, alpha=0.3)
ax.xaxis.set_major_locator(mticker.MultipleLocator(2))
plt.tight_layout()
plt.savefig('output/figures/fig03_industry_lev_equal_weight.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: fig03_industry_lev_equal_weight.png")


**图3解读**：金融业（J）负债率远高于其他行业（约0.85以上），与金融机构高杠杆经营模式一致，分析时应与非金融行业分开讨论。房地产（K）负债率在2015-2020年持续攀升，反映高杠杆扩张特征；建筑业（E）负债率也较高，与垫资施工的行业特征相符。

In [ ]:
def weighted_lev(grp):
    w = grp['total_assets']
    return (grp['Lev'] * w).sum() / w.sum()

aw_wt = ind_panel.groupby(['industry_code','year']).apply(weighted_lev).reset_index()
aw_wt.columns = ['industry_code','year','Lev_aw']

# 图4
fig, ax = plt.subplots(figsize=(12, 6))
for i, (code, label) in enumerate(target_inds.items()):
    sub = aw_wt[aw_wt['industry_code']==code].sort_values('year')
    ax.plot(sub['year'], sub['Lev_aw'], styles[i], color=colors[i], markersize=3, linewidth=1.5, label=f'{code}: {label}')
ax.set_xlabel('Year', fontsize=12)
ax.set_ylabel('Asset-Weighted Mean Lev', fontsize=12)
ax.set_title('Figure 4: Industry Leverage Ratio (Asset-Weighted)', fontsize=13)
ax.legend(fontsize=9, loc='upper left', bbox_to_anchor=(1,1))
ax.grid(True, alpha=0.3)
ax.xaxis.set_major_locator(mticker.MultipleLocator(2))
plt.tight_layout()
plt.savefig('output/figures/fig04_industry_lev_asset_weighted.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: fig04_industry_lev_asset_weighted.png")


**图4解读（两种算法比较）**：等权平均反映典型公司杠杆，加权平均反映典型资产杠杆，大公司权重更高。房地产行业加权平均明显高于等权平均，说明大型房企杠杆率显著高于中小房企，是债务风险的主要集中点。讨论宏观金融风险时，加权平均更合理，因大公司违约冲击更大。

## 第六部分：股权结构分析

In [ ]:
selected_years = [2001,2003,2005,2007,2009,2011,2013,2015,2017,2019,2021,2023]
top1_data = panel[panel['year'].isin(selected_years) & panel['Top1'].notna()]

fig, ax = plt.subplots(figsize=(14,6))
data_by_year = [top1_data[top1_data['year']==yr]['Top1'].values for yr in selected_years]
bp = ax.boxplot(data_by_year, labels=selected_years, patch_artist=True, notch=False,
                medianprops=dict(color='red', linewidth=2),
                flierprops=dict(marker='.', markersize=2, alpha=0.3))
for patch in bp['boxes']:
    patch.set_facecolor('lightblue')
    patch.set_alpha(0.7)
ax.set_xlabel('Year', fontsize=12)
ax.set_ylabel('Top1 Shareholding Ratio', fontsize=12)
ax.set_title('Figure 5: Boxplot of Top1 Shareholding Ratio (Selected Years)', fontsize=13)
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.savefig('output/figures/fig05_top1_boxplot_selected_years.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: fig05_top1_boxplot_selected_years.png")


In [ ]:
for yr in [2005, 2007, 2023]:
    sub = top1_data[top1_data['year']==yr]['Top1']
    print(f"{yr}: N={len(sub)}, Median={sub.median():.3f}, "
          f"Q1={sub.quantile(0.25):.3f}, Q3={sub.quantile(0.75):.3f}, "
          f"IQR={sub.quantile(0.75)-sub.quantile(0.25):.3f}, Max={sub.max():.3f}")


**图5解读**：2005年中位数约0.40，四分位距较小，反映股改前国有股高度集中且不流通的特征。股权分置改革（2005-2007年）完成后，2007年中位数略有下降，分布更分散，说明股权结构开始松动。2023年中位数降至约0.30，四分位距扩大，整体趋于分散，但仍有相当比例公司维持绝对控股（Top1 > 0.5）。仅凭Top1难以判断控制权稳定性，还需股东类型、一致行动人协议、董事会席位等补充信息。